Ключ, папка, модель

In [ ]:
YANDEX_CLOUD_MODEL = "yandexgpt"
YANDEX_CLOUD_API_KEY=""
YANDEX_CLOUD_FOLDER=""

Функции для извлечения полного текста веб-страницы (с лишними элементами)

In [ ]:
import requests
from bs4 import BeautifulSoup
from crawl4ai import AsyncWebCrawler
import trafilatura
import time

async def get_full_crawl4ai(url: str):
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url=url,
            word_count_threshold=10,
            # Попробуйте разные селекторы для вашего сайта:
            css_selector="article, .news-content, .post, [itemprop='articleBody']"
        )
        
        if result.success:
            # return {
            #     'title': result.metadata.get('title'),
            #     'content': result.markdown,
            #     # 'url': url
            # }
            return result.markdown
        return None
    
def get_full_html(url: str) -> str:
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        
        # максимально агрессивная чистка
        for tag in soup(["script", "style", "head", "meta", "link", "noscript"]):
            tag.decompose()
            
        text = soup.get_text(separator="\n", strip=True)
        lines = [line for line in text.splitlines() if len(line) > 30]
        full_line="\n".join(lines)
        print(full_line +"\n")
        return full_line
    except Exception as e:
        return f"Не удалось загрузить: {type(e).__name__} — {str(e)}"


def get_full_trafilatura(url, title_need=False):
    """Извлекает оригинальный заголовок и полный текст статьи"""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return None, None
            
        result = trafilatura.extract(
            downloaded,
            include_formatting=True,   # сохраняет абзацы
            include_links=False,
            include_comments=False,
            include_tables=False,
            no_fallback=True,
            output_format="txt"
        )
        
        # trafilatura часто может вытащить и заголовок отдельно
        metadata = trafilatura.extract_metadata(downloaded)
        original_title = metadata.title if metadata and metadata.title else None
        if title_need:
            return original_title
        else:
            return result
        
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")
        return None, None

# print(get_full_text("https://nao24.ru/obshestvo/45643-municipalnoe-predprijatie-severzhilkomservis-obespechit-rabotoj-v-letnij-period-bolee-90-podrostkov.html"))

4. Использование yandex_ai_studio_sdk для формирования списка новостей + Crawl4AI для извлечения текста + yandexgpt для чистого текста новостей и формирования таблицы

In [ ]:
from __future__ import annotations
import xml.etree.ElementTree as ET
import pathlib
import time
from typing import List, Dict, Literal, cast
from datetime import datetime
import pandas as pd
from pathlib import Path
import re

from yandex_ai_studio_sdk import AIStudio
from yandex_ai_studio_sdk.search_api import (
    FamilyMode,
    FixTypoMode,
    GroupMode,
    Localization,
    SortMode,
    SortOrder,
)

key=""
id=""

FOLDER_ID = id
API_KEY = key

USER_AGENT = (
    "Mozilla/5.0 (Linux; Android 13; Pixel 7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/125.0.6422.112 Mobile Safari/537.36"
)

DATE_FROM = "20250101"
DATE_TO = "20251231"

MAX_SAFE_PAGES = 200  # защита от бесконечного цикла
DELAY_BETWEEN_REQUESTS = 0.5  # задержка для API

In [7]:
# сбор новостей


def create_search_query(company_name):
    """
    Преобразует название компании в поисковый запрос
    Порядок действий:
    1. Сначала выделяем то, что в скобках
    2. Удаляем скобки из исходной строки
    3. Удаляем запятые и получаем основное название
    """
    
    # Оригинальная строка
    original = company_name.strip()
    
    # ШАГ 1: Находим всё, что в скобках
    brackets_pattern = r'\(([^)]+)\)'
    brackets_content = re.findall(brackets_pattern, original)
    
    # Обрабатываем содержимое скобок
    bracket_terms = []
    for content in brackets_content:
        # Очищаем содержимое скобок
        clean_content = content.strip()
        # Если в скобках несколько вариантов через запятую
        if ',' in clean_content:
            for item in clean_content.split(','):
                item_clean = item.strip()
                if item_clean:
                    # Убираем лишние пробелы внутри
                    item_clean = ' '.join(item_clean.split())
                    bracket_terms.append(item_clean)
        else:
            # Убираем лишние пробелы
            clean_content = ' '.join(clean_content.split())
            bracket_terms.append(clean_content)
    
    # ШАГ 2: Удаляем скобки с содержимым из исходной строки
    without_brackets = re.sub(r'\s*\([^)]*\)', '', original).strip()
    
    # ШАГ 3: Удаляем запятые и лишние пробелы
    # Сначала убираем запятые
    # without_commas = without_brackets.replace(',', ' ')
    # # Затем убираем лишние пробелы (множественные пробелы заменяем на один)
    # main_name = ' '.join(without_commas.split())

    main_name=without_brackets.split(',')[0].strip()
    
    # Формируем поисковый запрос
    search_parts = [f'"{main_name}"']
    
    # Добавляем термины из скобок
    for term in bracket_terms:
        if term and term != main_name:  # Избегаем дублирования
            search_parts.append(f'"{term}"')
    
    # Объединяем через "или"
    if len(search_parts) > 1:
        # Убираем дубликаты, сохраняя порядок
        unique_parts = []
        for part in search_parts:
            if part not in unique_parts:
                unique_parts.append(part)
        return ' | '.join(unique_parts)
    else:
        return search_parts[0]
    

# print(create_search_query("Магнит, АО (MGNT, МАГН)"))

def search(company, name, site):
    sdk = AIStudio(
        folder_id=FOLDER_ID,
        auth=API_KEY,
    )

    sdk.setup_default_logging()

    search = sdk.search_api.web("RU")

    search = search.configure(
        # family_mode=FamilyMode.STRICT,
        fix_typo_mode=FixTypoMode.OFF,
        # group_mode=GroupMode.DEEP,
        group_mode=GroupMode.FLAT,
        localization=Localization.RU,
        sort_mode=SortMode.BY_RELEVANCE,
        sort_order=SortOrder.DESC,
        user_agent=USER_AGENT,
        groups_on_page=100,
    )

    all_results: List[Dict] = []
    for month in range(1, 12, 1):
        new_name=create_search_query(name)
        # Формируем запрос
        search_query = (
            f"{new_name} "
            f"site:{site} "
            f"date:2025{month}01..2025{month+1}31"
        )

        print("Поисковый запрос:")
        print(search_query)
        print()

        for page in range(MAX_SAFE_PAGES):
            print(f"Запрос страницы {page}...")

            try:
                result_bytes = search.run(search_query, format="xml", page=page)
                result_text = result_bytes.decode("utf-8")
            except Exception as e:
                print("Ошибка запроса:", e)
                break

            root = ET.fromstring(result_text)
            # print(root)
            docs = root.findall(".//doc")
            # print(docs)

            if not docs:
                print("Результаты закончились.")
                break

            print(f"Найдено документов: {len(docs)}")

            timestamp = datetime.now().strftime('%Y-%m-%d')
            # Сохраняем XML страницы
            xml_filename = f"Страницы/{company}_{site.split('.')[0]}_{month}_2025_page_{page+1}_{timestamp}.xml"
            with open(xml_filename, "w", encoding="utf-8") as f:
                f.write(result_text)

            # Парсим результаты
            for doc in docs:
                url = doc.findtext("url")
                title = doc.findtext("title")
                if len(title)<=10 or "..." in title:
                    if len(title)<=10 or "..." in title and doc.findtext("extended-text"):
                        title=doc.findtext("extended-text")
                    else:
                        title=doc.findtext("headline")
                    
                mime = doc.findtext("mime-type")
                date_str=doc.findtext("modtime")
                dt = datetime.strptime(date_str, "%Y%m%dT%H%M%S")
                # Преобразуем в нужный формат
                formatted_date = dt.strftime("%d.%m.%Y")

                print(url)

                if url and mime!="pdf" and mime!="doc" and mime!="docx" and not("pdf" in url.lower()) and not("doc" in url.lower()) and not("xls" in url.lower()) and not("odt" in url.lower()):
                    all_results.append({
                        'NewsFinder': "парсер",
                        "CompanyName": company,
                        "NewsTitle": title,
                        'NewsDate': formatted_date,
                        'NewsSource': site[:-1],
                        "NewsURL": url,
                        "NewsText":"",
                        "CompanyVar": name
                    })

            time.sleep(DELAY_BETWEEN_REQUESTS)

    # Удаляем дубликаты
    unique_results = {item["NewsURL"]: item for item in all_results}.values()
    unique_results = list(unique_results)
    # print(type(unique_results), unique_results)

    print()
    print("Всего уникальных статей:", len(unique_results))

    return unique_results

# search('llm', '111', "habr.com/")

In [ ]:
# формируем таблицу с полным текстом новостей
import json
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

# ================= CONFIG =================

MODEL = f"gpt://{YANDEX_CLOUD_FOLDER}/yandexgpt"

async def formed_table(company, name, site):
    client = OpenAI(
        api_key=YANDEX_CLOUD_API_KEY,
        base_url="https://ai.api.cloud.yandex.net/v1",
        project=YANDEX_CLOUD_FOLDER
    )
    # ================= STEP 2 — DOWNLOAD ALL ARTICLES =================
    articles=search(company, name, site)
    articles_text = []
    urls=[]

    for i in articles:
        urls.append(i["NewsURL"])

        # i["NewsText"]=get_full_trafilatura(i["NewsURL"])
        
    for url in urls:
        try:
            articles_text.append(get_full_trafilatura(url))
            # articles_text.append(await extract_simple_news(url))
            
        except Exception as e:
            print("skip:", e)

    # ================= STEP 3 — STRUCTURE VIA LLM =================
    STRUCTURE_PROMPT = """
    Выдать ТОЛЬКО основной текст статьи.  
    Без рассуждений. Без объяснений. Без отказа.  
    Без "вот", "статья", "извлечено", "я не могу".  

    Начать сразу с первого слова текста статьи.  
    Закончить последним словом текста статьи.  
    Всё остальное игнорировать полностью."""


    for i in range(len(articles)):
        response = client.responses.create(
            model=MODEL,
            input=[
                {"role": "system", "content": STRUCTURE_PROMPT},
                {"role": "user", "content": f'{json.dumps(articles_text[i], ensure_ascii=False)}'}
            ],
        )

        print("\n✅ FINAL:\n")
        content=response.output_text.strip()
        if (content is not None) and content and not("null" in content.lower()):
            # print(content)
            try:
                articles[i]["NewsText"]=content
            except: pass
        else:
            text=await get_full_html(articles[i]["NewsURL"])
            response = client.responses.create(
            model=MODEL,
            input=[
                {"role": "system", "content": STRUCTURE_PROMPT},
                {"role": "user", "content": f'{json.dumps(text, ensure_ascii=False)}'}
            ],
            )
            print("\n✅ FINAL:\n")
            content=response.output_text.strip()
            try:
                articles[i]["NewsText"]=content
            except: pass

    print(articles)
    return articles

In [ ]:
# основной цикл. Чтение файла с компаниями
names = ["АО Новгородский водоканал", "АО ПКС-Водоканал", "ГОУП Мурманскводоканал", "ГП КО Водоканал"]

for i in range(len(names)):
    # формирование новостей по официальному названию
    name=names[i]
    print("name", name)
    # news_all=[]
    match name:
        case "АО Новгородский водоканал": sites=["novvedomosti.ru/", "53news.ru/", "novreg.ru/medianews/", "borovichanin.ru/", "pressa53.ru/", "vnnews.ru/", "vnru.ru/", "novgorod-tv.ru/"]
        case "АО ПКС-Водоканал": sites=["sampotv360.ru/", "gov10.ru/", "factornews.ru/", "stolicaonego.ru/", "ptzgovorit.ru/", "gubdaily.ru/", "karelia.news/", "tv-karelia.ru/category/novosti/", "karelinform.ru/"]
        case "ГОУП Мурманскводоканал": sites=["51.ru/", "nord-news.ru/", "novosti-murmanskoy-oblasti.ru/", "www.hibiny.ru/murmanskaya-oblast/news/", "severpost.ru/", "b-port.com/news/", "gov-murman.ru/info/news/", "murman.tv/ct-n-2--news/"]
        case "ГП КО Водоканал": sites=["kgd.ru/", "www.newkaliningrad.ru/", "ruwest.ru/", "klops.ru/", "gov39.ru/press/news/", "rugrad.online/news/"]

    for site in sites:
    # for site in ['adm-nao.ru/']:
        news= await formed_table(name, name, site)

        timestamp = datetime.now().strftime('%Y-%m-%d')
        site2=site.split(".")[0]
        csv_filename = f"Results with full text/{name}_{site2}_{timestamp}.csv"
        print(csv_filename)

        df = pd.DataFrame(news)
        print(df)
        try:
            df['NewsTitle'] = df['NewsTitle'].str.replace(r'"', "'", regex=True)
            df['NewsTitle'] = '"' + df['NewsTitle'] + '"'

            df['NewsText'] = df['NewsText'].str.replace(r'[\n\r\t]+', ' ', regex=True)
            # df['NewsTitle'] = df['NewsTitle'].str.replace(r'\s+', ' ', regex=True).str.strip()
            df['NewsText'] = df['NewsText'].str.replace(r'"', "'", regex=True)
            df['NewsText'] = '"' + df['NewsText'] + '"'
        except: pass
        
        df.to_csv(csv_filename, index=False, encoding="utf-8-sig")


name АО Новгородский водоканал
Поисковый запрос:
"АО Новгородский водоканал" site:novvedomosti.ru/ date:2025101..2025231

Запрос страницы 0...
Найдено документов: 16
https://novvedomosti.ru/news/society/110719/
https://novvedomosti.ru/news/society/109873/
https://novvedomosti.ru/news/economy/108057/
https://novvedomosti.ru/articles/region/57893/
https://novvedomosti.ru/news/society/107740/
https://novvedomosti.ru/news/society/103057/
https://novvedomosti.ru/news/society/103199/
https://novvedomosti.ru/news/economy/110174/
https://novvedomosti.ru/news/society/111714/
https://novvedomosti.ru/news/society/102834/
https://novvedomosti.ru/images/media/%D0%9D%D0%92%2025.02.26%20%E2%84%967.pdf
https://novvedomosti.ru/news/law/104853/
https://novvedomosti.ru/news/incidents/102654/
https://novvedomosti.ru/news/incidents/111771/
https://novvedomosti.ru/news/incidents/107076/
https://novvedomosti.ru/news/incidents/107002/
Запрос страницы 1...
Результаты закончились.
Поисковый запрос:
"АО Новгород